# Struktura danych

In [2]:
import requests
import time
import feedparser
import logging
from IPython.display import display, JSON, Markdown

# Ustawienie logów, żeby widzieć w notatniku, czy API odpowiada
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def fetch_arxiv_papers(query="cat:cs.LG+OR+cat:cs.AI", max_results=100):
    """
    Odpytuje API arXiv o najnowsze prace.
    Zoptymalizowane pod kątem unikania błędu 503.
    """
    base_url = "http://export.arxiv.org/api/query?"
    safe_query = query.replace(' ', '+')
    
    params = (
        f"search_query={safe_query}&"
        f"sortBy=submittedDate&"
        f"sortOrder=descending&"
        f"start=0&"
        f"max_results={max_results}"
    )
    
    max_retries = 3
    base_delay = 5
    
    for attempt in range(max_retries):
        try:
            headers = {
                'User-Agent': 'mailto:twoj.email@student.uczelnia.pl - ArxivRAG/1.0 (Polite Scraper)'
            }
            
            logger.info(f"Odpytywanie arXiv API (Próba {attempt+1}): {base_url}{params}")
            response = requests.get(base_url + params, headers=headers, timeout=30)
            
            if response.status_code == 200:
                time.sleep(3)
                return feedparser.parse(response.content)
            
            else:
                delay = base_delay * (2 ** attempt)
                logger.warning(f"arXiv zwrócił błąd {response.status_code}. Czekam {delay}s...")
                time.sleep(delay)
                
        except Exception as e:
            delay = base_delay * (2 ** attempt)
            logger.error(f"Błąd sieci: {e}. Czekam {delay}s...")
            time.sleep(delay)
            
    return None

In [5]:
import json

# Pobieramy 5 rekordów
feed = fetch_arxiv_papers(max_results=5)

if feed and hasattr(feed, 'entries') and feed.entries:
    # Konwertujemy rekordy na słowniki
    entries_data = [dict(entry) for entry in feed.entries[:5]]
    
    # Zapisujemy do pliku
    with open("wyniki_arxiv.json", "w", encoding="utf-8") as plik:
        json.dump(entries_data, plik, ensure_ascii=False, indent=4)
        
    print("✅ Zapisano pełne wyniki do pliku: wyniki_arxiv.json")
    print("Otwórz ten plik w edytorze, aby zobaczyć całą strukturę bez ucinania.")
else:
    print("Nie udało się pobrać danych lub feed jest pusty.")

INFO:__main__:Odpytywanie arXiv API (Próba 1): http://export.arxiv.org/api/query?search_query=cat:cs.LG+OR+cat:cs.AI&sortBy=submittedDate&sortOrder=descending&start=0&max_results=5


✅ Zapisano pełne wyniki do pliku: wyniki_arxiv.json
Otwórz ten plik w edytorze, aby zobaczyć całą strukturę bez ucinania.


## Oczyszczone dane do docelowej struktury

In [ ]:
import json
import pprint

# Przygotowujemy pustą listę na wyczyszczone dane
cleaned_data = []

if feed and hasattr(feed, 'entries') and feed.entries:
    for entry in feed.entries:
        # Tworzymy nowy słownik, przypisując tylko to, o co prosiłeś
        cleaned_entry = {
            "id": entry.get("id"),
            "link": entry.get("link"),
            "title": entry.get("title"),
            "updated": entry.get("updated"),
            "summary": entry.get("summary"),
            "published": entry.get("published"),
            
            # Magia Pythona (List comprehension): 
            # Przechodzimy przez każdy słownik w "tags" i wyciągamy z niego tylko "term"
            "tags": [tag.get("term") for tag in entry.get("tags", [])],
            
            # Wyciągamy konkretnie "term" ze słownika kategorii głównej
            "arxiv_primary_category": entry.get("arxiv_primary_category", {}).get("term"),
            
            # Podobnie jak z tagami, wyciągamy samo pole "name" dla każdego autora
            "authors": [author.get("name") for author in entry.get("authors", [])]
        }
        
        cleaned_data.append(cleaned_entry)

    # Zapisujemy wyczyszczone dane do pliku JSON
    with open("czyste_dane_arxiv.json", "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=4)
        
    print("✅ Dane zostały wyczyszczone i zapisane do pliku 'czyste_dane_arxiv.json'.")
    print("\n👀 Podgląd pierwszego rekordu po czyszczeniu:")
    
    # Wyświetlamy pierwszy rekord, żebyś od razu widział efekt
    pprint.pprint(cleaned_data[0], indent=2, width=100)
    
else:
    print("Brak danych do przetworzenia.")

✅ Dane zostały wyczyszczone i zapisane do pliku 'czyste_dane_arxiv.json'.

👀 Podgląd pierwszego rekordu po czyszczeniu:
{ 'arxiv_primary_category': 'cs.CL',
  'authors': [ 'Jan Tempus',
               'Philip Whittington',
               'Craig W. Schmidt',
               'Dennis Komm',
               'Tiago Pimentel'],
  'id': 'http://arxiv.org/abs/2605.22821v1',
  'link': 'https://arxiv.org/abs/2605.22821v1',
  'published': '2026-05-21T17:59:56Z',
  'summary': 'Tokenisation is an integral part of the current NLP pipeline. Current tokenisation '
             'algorithms such as BPE and Unigram are greedy algorithms -- they make locally '
             'optimal decisions without considering the resulting vocabulary as a whole. We '
             'instead formulate tokeniser construction as a linear program and solve it using '
             'convex optimisation tools, yielding a new algorithm we call ConvexTok. We find '
             'ConvexTok consistently improves intrinsic tokenisation

: 